# Mission 2: Digit Reader from Scratch 🔢

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/02_digit_reader.ipynb)

> **From your syllabus**
> - *ML-I*, Unit III (logistic regression) and Unit II (regularised least squares, bias-variance)
> - *Convex Optimization*, Unit II (convexity) and Unit V (linear classifiers)

In Mission 0 a library did the work. Now **you** build the digit reader with nothing but NumPy and the gradient descent you wrote in Mission 1.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its full code is in the cell, so you can read it. Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (run me first; the code is short, read it)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 2

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"0": {"title": "Ignite", "checkpoints": ["m00-ready", "m00-done"]}, "1": {"title": "Golf", "checkpoints": ["m01-grad", "m01-done"]}, "2": {"title": "Digit Reader", "checkpoints": ["m02-grad", "m02-done"]}, "3": {"title": "Selfie", "checkpoints": ["m03-ey", "m03-pca", "m03-done"]}, "4": {"title": "Mini-Google", "checkpoints": ["m04-markov", "m04-done"]}, "5": {"title": "Tiny AI", "checkpoints": ["m05-backprop", "m05-done"]}}
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, n) for n in range(len(MISSIONS))]
    for n, (st, got, tot) in enumerate(stats):
        x = 45 + n * 108
        title = MISSIONS[str(n)]["title"]
        if n:
            ok = stats[n - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">M{n} {title}</text>')
    finished = [n for n, t in enumerate(stats) if t[0] == "done"]
    todo = [f"M{n}" for n, t in enumerate(stats) if t[0] == "todo"]
    line = f"<b>{len(finished)} of {len(stats)} missions finished</b> &middot; now in <b>Mission {MISSION}</b> ({stats[MISSION][1]}/{stats[MISSION][2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else " &middot; every mission visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(0)
digits = load_digits()
X, y = digits.data / 16.0, digits.target     # scale pixels to [0, 1]
order = np.random.permutation(len(X))
train, test = order[:1300], order[1300:]
print(X[train].shape, X[test].shape)

## 1. Warm-up: is it a 0 or a 1?
Two classes first. Label $t_i=1$ if the digit is 1, else 0. The model gives the probability
$$p_i=\sigma(w^\top x_i+b),\qquad \sigma(z)=\frac1{1+e^{-z}}.$$
**Core 1:** fill in the sigmoid.

<details><summary>Hint</summary>

`np.exp(-z)` is $e^{-z}$.
</details>

In [ ]:
def sigmoid(z):
    return ____            # TODO: 1 / (1 + e^{-z})

print(sigmoid(np.array([-5.0, 0.0, 5.0])))   # expect about [0.007 0.5 0.993]

## 2. The loss and its gradient
Cross-entropy loss (ML-I Unit III), averaged over $n$ examples:
$$L(w,b)=-\frac1n\sum_i\big[t_i\log p_i+(1-t_i)\log(1-p_i)\big]$$

> **Math box.** Using $\sigma'(z)=\sigma(1-\sigma)$, the chain rule collapses beautifully to
> $$\nabla_w L=\frac1n X^\top(p-t),\qquad \frac{\partial L}{\partial b}=\frac1n\sum_i(p_i-t_i).$$
> $L$ is convex in $(w,b)$ (its Hessian $\frac1nX^\top\mathrm{diag}(p(1-p))X\succeq0$), so gradient descent cannot get stuck in a bad valley (Convex Optimization Unit II).

**Core 2:** fill in the gradient, then let the computer check your calculus.

In [ ]:
def loss(w, b, X, t):
    p = sigmoid(X @ w + b)
    eps = 1e-12
    return -np.mean(t * np.log(p + eps) + (1 - t) * np.log(1 - p + eps))

def gradient(w, b, X, t):
    p = sigmoid(X @ w + b)
    grad_w = ____          # TODO: (1/n) X^T (p - t)
    grad_b = ____          # TODO: mean of (p - t)
    return grad_w, grad_b

# Finite-difference check on random data
Xr, tr, wr = np.random.randn(20, 5), (np.random.rand(20) > 0.5) * 1.0, np.random.randn(5)
gw, gb = gradient(wr, 0.3, Xr, tr)
num = np.array([(loss(wr + 1e-5 * e, 0.3, Xr, tr) - loss(wr - 1e-5 * e, 0.3, Xr, tr)) / 2e-5 for e in np.eye(5)])
assert np.allclose(gw, num, atol=1e-6), "Gradient mismatch - recheck X^T (p - t) / n"
print("Gradient correct ✔")
checkpoint("m02-grad")

**Core 3:** the training loop. Same golf swing as Mission 1: step against the gradient.

In [ ]:
def fit(X, t, lr=1.0, steps=300):
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(steps):
        gw, gb = gradient(w, b, X, t)
        w = ____               # TODO: gradient descent update for w
        b = ____               # TODO: gradient descent update for b
        history.append(loss(w, b, X, t))
    return w, b, history

mask = (y[train] <= 1)
Xtr, ttr = X[train][mask], (y[train][mask] == 1) * 1.0
w, b, hist = fit(Xtr, ttr)
plt.plot(hist); plt.xlabel("stroke"); plt.ylabel("loss"); plt.show()

mt = (y[test] <= 1)
pred = sigmoid(X[test][mt] @ w + b) > 0.5
print("0-vs-1 test accuracy:", np.mean(pred == (y[test][mt] == 1)))

## 3. What did it learn?
The weight vector has 64 entries, one per pixel. Reshape it back to $8\times8$: red pixels say "this is a 1", blue say "this is a 0".

In [ ]:
plt.imshow(w.reshape(8, 8), cmap="coolwarm"); plt.colorbar(); plt.title("learned weights"); plt.show()

## 4. All ten digits: softmax
With 10 classes use one weight vector per class and $p_k=\dfrac{e^{z_k}}{\sum_j e^{z_j}}$. The gradient has the same shape:
$\nabla_W L=\frac1nX^\top(P-T)$ where $T$ is the one-hot label matrix. The code is provided; read it and notice how little changed.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # subtract the max for numerical stability
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def fit_softmax(X, labels, lr=1.0, steps=300, lam=0.0):
    T = np.eye(10)[labels]
    W, b = np.zeros((X.shape[1], 10)), np.zeros(10)
    for _ in range(steps):
        P = softmax(X @ W + b)
        W -= lr * (X.T @ (P - T) / len(X) + lam * W)
        b -= lr * (P - T).mean(axis=0)
    return W, b

def accuracy(W, b, X, labels):
    return np.mean(np.argmax(X @ W + b, axis=1) == labels)

W, b = fit_softmax(X[train], y[train])
print("train accuracy:", round(accuracy(W, b, X[train], y[train]), 3))
print("test accuracy :", round(accuracy(W, b, X[test], y[test]), 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W[:, k].reshape(8, 8), cmap="coolwarm"); ax.axis("off"); ax.set_title(f"digit {k}")
plt.tight_layout(); plt.show()

## Stretch: regularisation and bias-variance
Add the penalty $\frac\lambda2\|W\|^2$ (ridge, *ML-I Unit II*). Its gradient is $\lambda W$, already built into `fit_softmax` via `lam`.
Train on only 100 digits: train accuracy hits 100% but test lags, a sign of overfitting. Sweep $\lambda$ and **report honestly**: does it help here? Real experiments sometimes say no. Why might it not? (Think: how much data, how many steps; stopping early is also a form of regularisation.)

In [ ]:
small = train[:100]
for lam in [0.0, 0.003, 0.01, 0.03]:
    W2, b2 = fit_softmax(X[small], y[small], lam=lam, steps=500)
    print(f"lambda={lam:<6} train={accuracy(W2, b2, X[small], y[small]):.3f}  test={accuracy(W2, b2, X[test], y[test]):.3f}")

In [ ]:
checkpoint("m02-done")                      # Mission 2 complete: watch your map update at the top

## Quest (take-home)
Plot the **confusion matrix** of your classifier (10x10 counts of true vs predicted). Which two digits get confused most, and why does their weight image explain it? Show the 12 most confidently wrong pictures.

## You just used...
- the **sigmoid/softmax** model and **cross-entropy** loss (ML-I Unit III)
- the **chain rule** to get $\nabla L=\frac1nX^\top(p-t)$, and a finite-difference check
- **convexity** to trust gradient descent (Convex Optimization Unit II)
- **ridge regularisation** and the bias-variance trade-off (ML-I Unit II)

## Your CV line
> *Implemented multinomial logistic regression from scratch in NumPy (gradient derived and verified numerically), reaching about 95% test accuracy on handwritten digits; studied L2 regularisation and overfitting.*

Next: **Mission 3**, compress a selfie and find eigenfaces.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/03_selfie_compressor.ipynb)